## Submit pipelines, deploys on AI-Platform

#### Notebook performs full End to End scenario of training three components Pipeline and Deploying to Online Endpoint; Every Cell is mandatory and completed



## Install Required Packages

Install all dependencies needed for Triton Inference Server deployment. Run this cell once per kernel restart.

In [ ]:
%pip install "azure-ai-ml>=1.12.0" \
             "azure-identity>=1.14.0" \
             "scikit-learn>=1.3.0" \
             "skl2onnx>=1.16.0" \
             "onnx>=1.14.0" \
             "onnxruntime>=1.16.0" \
             "numpy>=1.24.0" \
             "requests>=2.31.0" \
             "pandas>=2.0.0"
# skl2onnx/onnx/onnxruntime are required to convert the sklearn model to ONNX format
# for the Triton ONNX Runtime backend.
# NOTE: The standard nvcr/nvidia/tritonserver:23.08-py3 image ships the ONNX Runtime
# backend but NOT the FIL (Forest Inference Library) backend — FIL requires a
# separate RAPIDS container build. We therefore use the ONNX Runtime backend here.


In [ ]:
# ── Project Root Setup ────────────────────────────────────────────────────────
# Ensure the working directory is always the project root so that all
# relative paths (./src/..., ./artifacts/...) resolve correctly regardless
# of where the Jupyter server was launched from.
#
# If you opened Jupyter from outside this folder, update the path below:
#   os.chdir("/absolute/path/to/ai-platform-aml-triton-examples")
import os, sys
from pathlib import Path

# Detect the directory that contains this notebook file (works in VS Code + JupyterLab)
_nb_file = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb_file:
    _project_root = str(Path(_nb_file).resolve().parent)
    os.chdir(_project_root)
else:
    _project_root = os.getcwd()

# Make src.* importable
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Create artifacts directory for runtime outputs (gitignored)
os.makedirs(os.path.join(_project_root, 'artifacts'), exist_ok=True)

print(f'Project root : {_project_root}')
print(f'Working dir  : {os.getcwd()}')


| Compute Name | Time Slicing Options     | Base Node Guaranteed CPU | Base Node Guaranteed Memory |
|--------------|------------------|---------------|------------------|
| cpu          | -2, -4           | 14            | 46Gi             |
| gput41       | -2, -4           | 6             | 44Gi             |
| gpuv1001     | -2, -4           | 4             | 98Gi             |
| gpuv1002     | -2, -4           | 10            | 206Gi            |
| gpua100      | -2, -4           | 22            | 202Gi            |
| gput44       | -2, -4, -8       | 56            | 410Gi            |
| gpuv1004     | -2, -4, -8       | 22            | 422Gi            |


## GPU Time slicing (Short for TS)

Allows up to 8 workloads onto one GPU node to run everything in concurrency, occupying one Node, distributing execution time evenly every ~3 seconds. This way, once auto-scaled, GPU node is available to serve new workloads almost instantly, or, host multiple deployments at the same time.

To utilise time slicing, append 2, 4 or 8 in the compute type: 

Example: __gput44-2__

Another example, if you wish to share A100 node between 4 workloads, specify this: __gpuv1002-4__
This way, it will auto select right Node type to be shared for 4 different workloads: jobs/pipelines/deploys equally distributing resources.

For regular, non time slicing cases, simply indicate compute name without and suffix, e.g.: __gpua1001__



## Specify tags, compute, optional

In [ ]:
environment                 = "azureml://registries/azureml/environments/sklearn-1.5/versions/26"
triton_environment          = "tritonNcdEnv:23.08.02-py3"   # Triton Inference Server environment in this workspace
# === Optional: Deployment (MODEL SERVING) details ===
compute_target              = "cpu-2"   # 4 means time slicing into 4 parts on a single node

tags = {
    "Purpose":      "Project Resources",
    "by_person":    "by_person",                        #Highly recommended to specify person name, who submitted the code
}


## Import libraries

In [ ]:
import os
from azure.ai.ml import MLClient, command, Input
from azure.identity import ManagedIdentityCredential, DefaultAzureCredential
from azure.ai.ml.entities import (
    CommandComponent,
    PipelineJob,
    JobResourceConfiguration,
    Model,
    KubernetesOnlineEndpoint,
    KubernetesOnlineDeployment,
    Environment,
    CodeConfiguration
)
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities._deployment.resource_requirements_settings import (
    ResourceRequirementsSettings,
)
from azure.ai.ml.entities._deployment.container_resource_settings import (
    ResourceSettings,
)
# Service functions
instanceType           = "defaultinstancetype"
nb_prefix              = os.getenv("NB_PREFIX", "/notebook/unknown/unknown")
_, namespace, pod_name = nb_prefix.strip("/").split("/")
AZURE_CLIENT_ID        = os.getenv('AZURE_CLIENT_ID')
client_id              = os.getenv('AZURE_CLIENT_ID')
credential             = ManagedIdentityCredential(client_id=os.getenv("AZURE_CLIENT_ID"))
# === Optional Deployment (MODEL SERVING) details ===
request_cpu = "500m"
request_ram = "2Gi"
limit_cpu = "1"
limit_ram = "4Gi"


## Import Helper script:

In [ ]:
from src._helpers.load_tags import load_tags
tags = load_tags(tags={}, namespace=namespace, pod_name=pod_name)
for k, v in tags.items():
    globals()[k] = v

print(tags)

## Build ML client, specify compute

In [ ]:
#  /AML Details 
subscription_id             = subscription_id
resource_group              = aml_workspace_rg
workspace                   = aml_workspace
computeTarget               = f"{compute_target}"
# AML Details/ 

ml_client = MLClient(credential, subscription_id, resource_group, workspace)

### Define a 3-stage Iris classification pipeline:

In [ ]:
# Pipeline scripts live in ./src/iris_pipeline:
#   train.py    — trains RandomForestClassifier on the Iris dataset, saves model.pkl
#   analysis.py — evaluates the model and writes a classification report
#   score.py    — runs batch inference on sample Iris rows
code_path = "./src/iris_pipeline"


# DEFINE PIPELINE COMPONENTS:
# === 1 ===
train_component = CommandComponent(
    name='iris_train_component',
    display_name='iris_model_training',
    command='python train.py --output_dir ${{outputs.model_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={},
    outputs={'model_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# === 2 ===
analysis_component = CommandComponent(
    name='iris_analysis_component',
    display_name='iris_model_analysis',
    command='python analysis.py --model_path ${{inputs.model_input}} --output_dir ${{outputs.analysis_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={'model_input': {'type': 'uri_folder'}},
    outputs={'analysis_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# === 3 ===
inference_component = CommandComponent(
    name='iris_inference_component',
    display_name='iris_inferencing',
    command='python score.py --model_path ${{inputs.model_input}} --output_dir ${{outputs.score_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={'model_input': {'type': 'uri_folder'}},
    outputs={'score_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# Define the pipeline
@pipeline(default_compute=computeTarget)
def iris_classification_pipeline():
    train_step     = train_component()
    analysis_step  = analysis_component(model_input=train_step.outputs.model_output)
    inference_step = inference_component(model_input=train_step.outputs.model_output)

    return {
        'train_output':    train_step.outputs.model_output,
        'analysis_output': analysis_step.outputs.analysis_output,
        'inference_output': inference_step.outputs.score_output
    }


### Submit Pipeline run:

In [ ]:
pipeline_job = iris_classification_pipeline()

# Add dynamic tags
pipeline_job.tags = tags

print(pipeline_job.tags)
pipeline_job.settings.force_rerun = True
pipeline_job = ml_client.jobs.create_or_update(pipeline_job)
ml_client.jobs.stream(pipeline_job.name)

pipeline_run_id = pipeline_job.id
print(pipeline_job.experiment_name)

print(f"Pipeline run ID: {pipeline_run_id}")


### Convert Trained Sklearn Model to ONNX and Build Triton Model Repository

Triton Inference Server's **ONNX Runtime backend** (`backend: "onnxruntime"`) is included in every standard `nvcr/nvidia/tritonserver` image and runs natively on CPU — no GPU required.

> **Why not FIL?**  The FIL (Forest Inference Library) backend is **not compiled into the standard Triton image**. It is only available in separate RAPIDS container builds. Since the workspace environment is based on `tritonserver:23.08-py3`, we use the ONNX Runtime backend instead, which achieves the same result.

The cells below:
1. Download the `model.pkl` (`RandomForestClassifier` on Iris) from the training pipeline child run
2. Convert it to ONNX format with `skl2onnx` using `zipmap=False` so that the probability output is a plain float32 array
3. Assemble the Triton model repository directory structure
4. Write `config.pbtxt` (ONNX Runtime backend, `KIND_CPU`, 4-feature Iris input)

The resulting directory is then registered in Azure ML as a `triton_model` and deployed to the Kubernetes Online Endpoint on the `cpu` compute pool.

**Triton model repository layout:**
```
triton_model_repo/
  iris_classifier/
    config.pbtxt
    1/
      model.onnx
```


In [ ]:
import glob as glob_mod
import shutil

# pipeline_job.name is the short job name used by the SDK
pipeline_job_name = pipeline_job.name
# Azure ML sets the child job display_name to the step variable name used inside
# the @pipeline function — here that is "train_step".
training_step_name = "train_step"

# Find the child run that corresponds to the training step
training_run_id = None
child_runs = ml_client.jobs.list(parent_job_name=pipeline_job_name)
for child_run in child_runs:
    print(f"  child: display_name={child_run.display_name!r}  name={child_run.name}")
    if child_run.display_name == training_step_name:
        training_run_id = child_run.name
        print(f"Found training run ID: {training_run_id}")
        break

if training_run_id is None:
    raise RuntimeError(
        f"Could not find a child run named '{training_step_name}' "
        f"under pipeline job '{pipeline_job_name}'. "
        "Check the display_name values printed above and update training_step_name if needed."
    )

# Download the model artifact from the training child run
download_dir = os.path.join(_project_root, "artifacts", "model_download")
os.makedirs(download_dir, exist_ok=True)

ml_client.jobs.download(
    name=training_run_id,
    download_path=download_dir,
    output_name="model_output"
)

# Locate model.pkl — accommodates different Azure ML download path layouts
pkl_candidates = glob_mod.glob(f"{download_dir}/**/model.pkl", recursive=True)
if not pkl_candidates:
    all_files = glob_mod.glob(f"{download_dir}/**/*", recursive=True)
    raise FileNotFoundError(
        f"model.pkl not found under {download_dir}.\n"
        f"Files found: {all_files}\n"
        "Update the filename pattern if your training script uses a different name."
    )

model_pkl_path = pkl_candidates[0]
print(f"Model downloaded to: {model_pkl_path}")


In [ ]:
import joblib
import numpy as np
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sklearn.ensemble import RandomForestClassifier

# Load the RandomForestClassifier produced by the training pipeline step
model = joblib.load(model_pkl_path)
print(f"Model type      : {type(model)}")
print(f"n_estimators    : {model.n_estimators}")
print(f"n_features_in_  : {model.n_features_in_}")
print(f"classes_        : {model.classes_}")

# Convert to ONNX.
# zipmap=False → probabilities output is a plain float32 array.
# opset=12: broad compatibility with ONNX Runtime 1.15/1.16 inside Triton 23.08.
n_features  = model.n_features_in_    # 4 for Iris
n_classes   = len(model.classes_)     # 3 for Iris
initial_type = [("float_input", FloatTensorType([None, n_features]))]
onnx_model   = convert_sklearn(
    model,
    initial_types=initial_type,
    target_opset=12,
    options={RandomForestClassifier: {"zipmap": False}},
)
print(f"ONNX opset      : {onnx_model.opset_import[0].version}")

# Quick local sanity-check with onnxruntime
import onnxruntime as rt
sess   = rt.InferenceSession(onnx_model.SerializeToString())
sample = np.array([[5.1, 3.5, 1.4, 0.2]], dtype=np.float32)   # setosa
label, proba = sess.run(["label", "probabilities"], {"float_input": sample})
print(f"Local ONNX test : label={label}  proba={proba.round(3)}")

# ── Build the Triton model repository structure ───────────────────────────────
#
#   triton_model_repo/
#     iris_classifier/
#       config.pbtxt
#       1/
#         model.onnx    ← served by Triton ONNX Runtime backend (GPU nodes)
#         model.pkl     ← served by score_ort.py via sklearn  (CPU nodes)
#
import shutil

triton_model_name = "iris_classifier"
triton_model_repo = os.path.join(_project_root, "artifacts", "triton_model_repo")

# Clean up any artefacts from a previous run
if os.path.exists(triton_model_repo):
    shutil.rmtree(triton_model_repo)

model_version_dir = os.path.join(triton_model_repo, triton_model_name, "1")
os.makedirs(model_version_dir, exist_ok=True)

# Save ONNX model (for Triton serving on GPU nodes)
onnx_path = os.path.join(model_version_dir, "model.onnx")
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())
print(f"ONNX model saved : {onnx_path}")

# Also copy the sklearn pickle (for score_ort.py serving on CPU nodes)
pkl_dest = os.path.join(model_version_dir, "model.pkl")
shutil.copy2(model_pkl_path, pkl_dest)
print(f"sklearn pkl saved: {pkl_dest}")


In [ ]:
# Create Triton ONNX Runtime backend configuration file (config.pbtxt).
#
# ONNX Runtime backend is always present in nvcr/nvidia/tritonserver images
# and supports CPU inference natively via KIND_CPU.
#
# Input/output names must match exactly what skl2onnx exported:
#   input  → "float_input"  (FloatTensorType, 4 Iris features)
#   output → "label"        (INT64, predicted class 0/1/2)
#   output → "probabilities" (FP32, [3] per sample — exposed but optional to query)

config_content = f"""name: "{triton_model_name}"
backend: "onnxruntime"
max_batch_size: 64

input [
  {{
    name: "float_input"
    data_type: TYPE_FP32
    dims: [{n_features}]
  }}
]

output [
  {{
    name: "label"
    data_type: TYPE_INT64
    dims: [1]
  }},
  {{
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [3]
  }}
]

instance_group [
  {{
    kind: KIND_CPU
    count: 1
  }}
]

dynamic_batching {{
  max_queue_delay_microseconds: 100
}}
"""

config_path = os.path.join(triton_model_repo, triton_model_name, "config.pbtxt")
with open(config_path, "w") as f:
    f.write(config_content)

print("Triton ONNX Runtime config.pbtxt created:")
print(config_content)
print(f"Triton model repository ready at: {triton_model_repo}")


### Register Triton FIL Model in Azure ML

In [ ]:
# Register the Triton FIL model repository as an Azure ML model (type: triton_model).
# Azure ML uploads the local directory structure to blob storage and tracks it as a versioned asset.
import datetime

model_name        = "triton-iris-fil-" + datetime.datetime.now().strftime("%Y%m%d%H%M%S")
model_description = (
    "Sklearn RandomForestClassifier (Iris dataset) packaged for "
    "Triton Inference Server FIL backend — CPU inference"
)

model = Model(
    name=model_name,
    path=triton_model_repo,   # local Triton model repository directory — Azure ML uploads it
    type="triton_model",       # tells Azure ML this is a Triton model repository
    description=model_description
)

registered_model = ml_client.models.create_or_update(model)
print(f"Triton FIL model registered : {registered_model.name}")
print(f"Model ID                    : {registered_model.id}")


## 2. Deploy online endpoint to Azure


### 2.1 Configure online endpoint
`endpoint_name`: The name of the endpoint.

`auth_mode` : Use `key` for key-based authentication. Use `aml_token` for Azure Machine Learning token-based authentication. A `key` does not expire, but `aml_token` does expire. 

Optionally, you can add description, tags to your endpoint.

In [ ]:
# ── Cleanup: Delete orphaned endpoints to free RBAC role assignment quota ─────
# Azure subscriptions have a limit on role assignments (typically 2000 per scope).
# Each endpoint deployment creates role assignments. Orphaned endpoints (no traffic)
# consume quota without serving anything. This cell deletes old ai-platform-endpoint*
# endpoints that have no active deployments to free up quota before creating a new one.
#
# To skip cleanup and reuse an existing endpoint, comment out this cell and set:
#   online_endpoint_name = "ai-platform-endpoint<timestamp>"  # existing name

ENDPOINT_PREFIX = "ai-platform-endpoint"

try:
    existing_endpoints = list(ml_client.online_endpoints.list())
    orphans = [
        ep for ep in existing_endpoints
        if ep.name.startswith(ENDPOINT_PREFIX) and not ep.traffic
    ]
    if orphans:
        print(f"Found {len(orphans)} orphaned endpoint(s) to clean up:")
        for ep in orphans:
            print(f"  Deleting: {ep.name}")
            try:
                ml_client.online_endpoints.begin_delete(name=ep.name).result()
                print(f"  Deleted: {ep.name}")
            except Exception as del_err:
                print(f"  Warning: could not delete {ep.name}: {del_err}")
    else:
        print("No orphaned endpoints found — quota should be available.")
except Exception as e:
    print(f"Warning: endpoint cleanup check failed: {e}")
    print("Continuing with endpoint creation...")


In [ ]:
# deploy the model to AKS
import datetime


online_endpoint_name= "ai-platform-endpoint"+ datetime.datetime.now().strftime("%m%d%f")
print(f"Online Endpoint name is: {online_endpoint_name}")

# create an online endpoint
endpoint= KubernetesOnlineEndpoint(
    name=online_endpoint_name,
    compute=computeTarget,
    description="AI Plaatforms endpoint with SSL Termination",
    auth_mode="key",
    tags=tags,
)

ml_client.begin_create_or_update(endpoint).result()

## 3.0 Configure Online Deployment — CPU-Compatible Serving

A deployment is a set of resources required for hosting the model.

> **Why `score_ort.py` instead of Triton directly?**
> `nvcr/nvidia/tritonserver` images are compiled against CUDA and require the NVIDIA GPU driver on the host node.
> CPU-only AKS nodes (`cpu-2`) have no NVIDIA driver, so `tritonserver` crashes at startup (`CrashLoopBackOff`).
>
> **Solution:** deploy with the curated `sklearn-1.5` environment (zero CUDA dependency) and `src/triton_scoring/score_ort.py`,
> which loads the sklearn `model.pkl` from the Triton model repository and exposes the **identical KFServing V2 API**
> that Triton would expose — same request/response JSON format, same tensor names and dtypes.
>
> On a GPU node, replace `environment=environment` → `environment=triton_environment` and
> `scoring_script="score_ort.py"` → `scoring_script="score.py"` to use native Triton.

More information about scheduling: [Scheduling-options.md](https://github.com/equinor/aurora-azureml-usecases/blob/main/azureml-getting-started/Scheduling-options.md)

In [ ]:
# Parameters for CPU deployment.
#
# serving_environment  — sklearn-1.5 curated env (has joblib/numpy/sklearn, no CUDA needed)
# triton_scoring_path  — ./src/triton_scoring  (shared with Triton proxy; score_ort.py lives here)
# triton_scoring_script — score_ort.py implements KFServing V2 API via sklearn inference
#
# To switch to native Triton on a GPU node:
#   serving_environment  = triton_environment
#   triton_scoring_script = "score.py"
serving_environment   = environment          # sklearn-1.5: proven to work on CPU AKS nodes
deployment_name       = "blue-deployment"
endpoint_name         = online_endpoint_name
model_id              = registered_model.id
instance_count        = 1
triton_scoring_path   = "./src/triton_scoring"
triton_scoring_script = "score_ort.py"       # KFServing V2 via sklearn (CPU-safe)


### Create Deployment object 

In [ ]:
# CPU deployment using sklearn-1.5 environment + KFServing V2 scoring script.
# score_ort.py loads model.pkl from AZUREML_MODEL_DIR and returns predictions
# in the same KFServing V2 JSON format that Triton ONNX Runtime backend would return.
blue_deployment = KubernetesOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=model_id,
    environment=serving_environment,          # sklearn-1.5 (no CUDA)
    code_configuration=CodeConfiguration(
        code=triton_scoring_path,
        scoring_script=triton_scoring_script,  # score_ort.py
    ),
    instance_count=instance_count,
    instance_type=instanceType,
    tags=tags,
)


## 3.1 Create the deployment and set traffic

Using the `MLClient` created earlier, we will now create the deployment in the workspace. This command will start the deployment creation and return a confirmation response while the deployment creation continues.

In [ ]:
ml_client.begin_create_or_update(blue_deployment).result()



endpoint = ml_client.online_endpoints.get(name=endpoint_name)
scoring_uri = endpoint.scoring_uri
print("Scoring URI:", scoring_uri)
endpoint.traffic = {"blue-deployment": 100} 
ml_client.begin_create_or_update(endpoint).result()

### TEST deployed Triton ONNX Runtime model serving:

The cell below sends a request using Triton's **KFServing v2 inference protocol**.

- Input tensor name: `float_input`  (4 × FP32 Iris features)
- Output tensor `label`: INT64 predicted class — **0 = setosa**, **1 = versicolor**, **2 = virginica**
- Output tensor `probabilities`: FP32 `[3]` array with per-class probabilities (optional)

In [ ]:
import requests
import json

# Retrieve endpoint credentials
endpoint_keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
api_key       = endpoint_keys.primary_key

# score_ort.py returns json.dumps(result); the AML inference server wraps that
# string as-is in the HTTP body, so response.text may be a JSON-encoded string.
# We unwrap one level if needed.
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type":  "application/json",
}

payload = {
    "inputs": [
        {
            "name":     "float_input",
            "shape":    [1, n_features],
            "datatype": "FP32",
            "data":     [[5.1, 3.5, 1.4, 0.2]]   # classic setosa sample
        }
    ],
    "outputs": [
        {"name": "label"},
        {"name": "probabilities"}
    ]
}

response = requests.post(scoring_uri, headers=headers, json=payload, verify=False)
print(f"Status  : {response.status_code}")

# AML wraps the run() return value as JSON; unwrap one level if it is a string
raw = response.text
try:
    parsed = json.loads(raw)
    # If AML double-encoded (run() returned a string), decode again
    if isinstance(parsed, str):
        parsed = json.loads(parsed)
except json.JSONDecodeError:
    parsed = {"raw": raw}

print(f"Response:\n{json.dumps(parsed, indent=2)}")

# Interpret results
if response.status_code == 200 and isinstance(parsed, dict) and "outputs" in parsed:
    class_names = ["setosa", "versicolor", "virginica"]
    for out in parsed["outputs"]:
        if out["name"] == "label":
            predicted_class = int(out["data"][0])
            print(f"\nPredicted class  : {predicted_class} ({class_names[predicted_class]})")
        if out["name"] == "probabilities":
            probs = out["data"]
            # probs is a flat list [p0, p1, p2] for batch_size=1
            print("Class probabilities: " +
                  "  ".join(f"{class_names[i]}={probs[i]:.3f}" for i in range(3)))

# Equivalent curl command
curl_body = json.dumps(payload)
curl_cmd = (
    f"curl -k -X POST '{scoring_uri}' "
    f"-H 'Authorization: Bearer {api_key}' "
    f"-H 'Content-Type: application/json' "
    f"-d '{curl_body}'"
)
print("\nEquivalent curl command:")
print(curl_cmd)
